# How reliable is genomic prediction within a rice breeding programme?
### Across-family accuracy, explainable AI (SHAP) and conformal prediction intervals with 1k-RiCA SNP data

**Data:** 353 elite *indica* breeding lines (24 families), 941 SNPs after QC, traits FLW (days to flowering), GY (grain yield) and PH (plant height). Source: Arbelaez et al. (2019) *Rice* 12:55, https://doi.org/10.1186/s12284-019-0311-0

**How to run:** upload `1kRiCA_clean_data.xlsx` to Colab (Files panel), then **Runtime → Run all**. Total run time is about 10 minutes. Keep the tab open while it runs. Cell 6 downloads every figure and table.

| Cell | What it does |
|---|---|
| 1 | Load data; GBLUP, Bayesian ridge, random forest, XGBoost; random 5-fold CV vs leave-one-family-out (LOFO) |
| 2 | SHAP: beeswarm, heatmap by family, genome-wide importance, SHAP stability across families |
| 3 | Conformal prediction intervals: random vs family-aware calibration |
| 4 | Circular tree, raincloud, observed vs predicted, t-SNE, ridgeline, SHAP dependence, forest plot |
| 5 | 10×5 repeated CV, within-family and parent-disjoint LOFO, Wilcoxon tests, sensitivity checks, genes near top SHAP SNPs |
| 5b | Gene lookup with retries |
| 7 | Candidate-gene figure (Ghd8, sd1) |
| 6 | Download all outputs |


## Cell 1 — Data and models

In [ ]:
# ===== CELL 1: Data + models: random CV vs new family (~2 min) =====
import numpy as np, pandas as pd, time
from scipy import stats, optimize
from sklearn.model_selection import KFold
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# ---- 1. Load data (finds the file automatically; asks to upload if missing) ----
import glob, os
found = [f for f in glob.glob("/content/**/*.xlsx", recursive=True) if "1kRiCA" in os.path.basename(f)]
if found:
    PATH = found[0]
else:
    from google.colab import files
    PATH = list(files.upload().keys())[0]
print("Using:", PATH)
sheets = pd.ExcelFile(PATH).sheet_names
if "data" in sheets:                                   # 1kRiCA_clean_data.xlsx
    df = pd.read_excel(PATH, sheet_name="data")
    snp_cols = list(df.columns[12:])
else:                                                  # 1kRiCA_GS_clean_dataset.xlsx
    lines = pd.read_excel(PATH, sheet_name="lines")
    geno = pd.read_excel(PATH, sheet_name="genotypes_012")
    lines["id"] = lines["id"].astype(str); geno["id"] = geno["id"].astype(str)
    df = lines.merge(geno, on="id")
    snp_cols = [c for c in geno.columns if c != "id"]
X = df[snp_cols].to_numpy(float)                       # 941 SNPs (0/1/2)
families = df["family"].astype(str).to_numpy()
TRAITS = ["FLW", "GY", "PH"]
print(X.shape, "| families:", len(set(families)))

# ---- 2. Models ----
class GBLUP:
    """Genomic BLUP fitted by REML (same as rrBLUP::mixed.solve)."""
    def fit(self, X, y):
        self.p = X.mean(0) / 2; self.c = 2 * np.sum(self.p * (1 - self.p)); self.X = X
        K = (X - 2 * self.p) @ (X - 2 * self.p).T / self.c
        D, U = np.linalg.eigh(K); D = np.clip(D, 0, None); yt, xt = U.T @ y, U.T @ np.ones(len(y))
        def nll(ld):
            w = 1 / (D + np.exp(ld)); b = (xt * w @ yt) / (xt * w @ xt); r = yt - xt * b
            return 0.5 * ((len(y) - 1) * np.log(r * w @ r) - np.log(w).sum() + np.log(xt * w @ xt))
        d = np.exp(optimize.minimize_scalar(nll, bounds=(-10, 10), method="bounded").x)
        V = K + d * np.eye(len(y)); v1 = np.linalg.solve(V, np.ones(len(y)))
        self.mu = v1 @ y / v1.sum(); self.a = np.linalg.solve(V, y - self.mu); self.h2 = 1 / (1 + d)
        return self
    def predict(self, Xn):
        return self.mu + (Xn - 2 * self.p) @ (self.X - 2 * self.p).T / self.c @ self.a

def get_model(name):
    return {"GBLUP": lambda: GBLUP(),
            "BayesianRidge": lambda: BayesianRidge(),
            "RandomForest": lambda: RandomForestRegressor(n_estimators=200, max_features=0.33,
                                                          min_samples_leaf=3, n_jobs=-1, random_state=1),
            "XGBoost": lambda: XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                                            colsample_bytree=0.5, tree_method="hist", n_jobs=-1, random_state=1)}[name]()
MODELS = ["GBLUP", "BayesianRidge", "RandomForest", "XGBoost"]

def fit_predict(name, Xtr, ytr, Xte):
    mu, sd = ytr.mean(), ytr.std()                  # scale inside training set only
    return get_model(name).fit(Xtr, (ytr - mu) / sd).predict(Xte) * sd + mu

# ---- 3. Evaluate: random 5-fold CV vs leave-one-family-out (LOFO) ----
big_fams = [f for f, c in pd.Series(families).value_counts().items() if c >= 15]
results = []; t0 = time.time()
for trait in TRAITS:
    y = df[trait].to_numpy(float)
    for name in MODELS:
        r_cv = [stats.pearsonr(fit_predict(name, X[tr], y[tr], X[te]), y[te])[0]
                for tr, te in KFold(5, shuffle=True, random_state=1).split(X)]
        r_lofo = [stats.pearsonr(fit_predict(name, X[families != f], y[families != f], X[families == f]),
                                 y[families == f])[0] for f in big_fams]
        results.append(dict(trait=trait, model=name, r_randomCV=np.mean(r_cv), r_LOFO=np.mean(r_lofo)))
        print(f"{trait:4s} {name:14s} random CV r = {np.mean(r_cv):.3f} | new family (LOFO) r = {np.mean(r_lofo):.3f}"
              f"  [{time.time()-t0:.0f}s]")

res = pd.DataFrame(results).round(3)
print("\n", res.to_string(index=False))
res.to_csv("/content/model_results.csv", index=False)
print("Saved /content/model_results.csv")

## Cell 2 — Explainable AI (SHAP)

In [ ]:
# ===== CELL 2: SHAP + publication figures (run after the model cell, same session) =====
import matplotlib.pyplot as plt, matplotlib as mpl, xgboost as xgb
mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "axes.titleweight": "bold"})
C = ["#0072B2", "#E69F00", "#009E73", "#D55E00"]                     # colour-blind safe
chrom = np.array([int(s.split(":")[0].replace("chr", "")) for s in snp_cols])
pos = np.array([int(s.split(":")[1]) for s in snp_cols])
off = dict(zip(range(1, 13), np.cumsum([0] + [pos[chrom == c].max() for c in range(1, 12)])))
gpos = (pos + np.array([off[c] for c in chrom])) / 1e6

def shap_values(Xfit, yfit):
    m = get_model("XGBoost").fit(Xfit, (yfit - yfit.mean()) / yfit.std())
    return m.get_booster().predict(xgb.DMatrix(Xfit), pred_contribs=True)[:, :-1]

# ---- Figure A: random CV vs new family, all models ----
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)
for i, t in enumerate(TRAITS):
    d = res[res.trait == t].set_index("model").loc[MODELS]
    yy = np.arange(len(MODELS))
    ax[i].hlines(yy, d.r_LOFO, d.r_randomCV, color="#BBBBBB", lw=2)
    ax[i].scatter(d.r_randomCV, yy, s=60, color=C[0], label="Random CV", zorder=3)
    ax[i].scatter(d.r_LOFO, yy, s=60, color=C[3], label="New family (LOFO)", zorder=3)
    ax[i].axvline(0, c="k", lw=0.6, ls=":"); ax[i].set_yticks(yy, MODELS); ax[i].invert_yaxis()
    ax[i].set_title(t); ax[i].set_xlabel("Prediction accuracy (r)")
fig.legend(*ax[0].get_legend_handles_labels(), loc="upper right", ncol=2, fontsize=8)
fig.suptitle("Accuracy drops when predicting a new family", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/FigA_accuracy.png", bbox_inches="tight"); plt.show()

# ---- SHAP for each trait: beeswarm + heatmap by family + genome-wide Manhattan ----
top_tables = []
for t in TRAITS:
    y = df[t].to_numpy(float); S = shap_values(X, y)
    imp = np.abs(S).mean(0); idx = np.argsort(-imp)[:20]
    top_tables.append(pd.DataFrame({"trait": t, "SNP": np.array(snp_cols)[idx], "chrom": chrom[idx],
                                    "pos": pos[idx], "mean_abs_SHAP": imp[idx]}))
    fig = plt.figure(figsize=(16, 9)); g = fig.add_gridspec(2, 2, width_ratios=[1, 1.5], height_ratios=[1.4, 0.7])
    # beeswarm
    ax = fig.add_subplot(g[0, 0]); rng = np.random.default_rng(1)
    for r, j in enumerate(idx[::-1]):
        ax.scatter(S[:, j], r + rng.uniform(-0.3, 0.3, len(S)), c=X[:, j], cmap="coolwarm", vmin=0, vmax=2, s=6, lw=0)
    ax.axvline(0, c="grey", lw=0.6); ax.set_yticks(range(20), np.array(snp_cols)[idx[::-1]], fontsize=7)
    ax.set_xlabel("SHAP value (impact on prediction)"); ax.set_title("a  Top 20 SNPs (XGBoost SHAP)", loc="left")
    plt.colorbar(mpl.cm.ScalarMappable(cmap="coolwarm", norm=mpl.colors.Normalize(0, 2)), ax=ax,
                 ticks=[0, 1, 2], fraction=0.04).set_label("Minor allele dosage")
    # heatmap: lines grouped by family
    ax = fig.add_subplot(g[0, 1]); o = np.lexsort((-S.sum(1), families)); v = np.percentile(np.abs(S[:, idx]), 99)
    im = ax.imshow(S[np.ix_(o, idx)], aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, interpolation="nearest")
    fs = families[o]
    for e in np.where(fs[1:] != fs[:-1])[0] + 0.5: ax.axhline(e, c="k", lw=0.3)
    lab = [(np.where(fs == f)[0].mean(), f) for f in pd.unique(fs) if (fs == f).sum() >= 10]
    ax.set_yticks([a for a, _ in lab], [b for _, b in lab], fontsize=7)
    ax.set_xticks(range(20), np.array(snp_cols)[idx], rotation=90, fontsize=6)
    ax.set_title("b  SHAP heatmap: lines grouped by family", loc="left"); plt.colorbar(im, ax=ax, fraction=0.03).set_label("SHAP")
    # Manhattan
    ax = fig.add_subplot(g[1, :])
    ax.scatter(gpos, imp, c=np.where(chrom % 2 == 0, C[0], "#9AB7D3"), s=10, lw=0)
    for j in idx[:5]: ax.annotate(snp_cols[j], (gpos[j], imp[j]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xticks([off[c] / 1e6 + pos[chrom == c].max() / 2e6 for c in range(1, 13)], [f"Chr{c}" for c in range(1, 13)])
    ax.set_ylabel("Mean |SHAP|"); ax.set_title("c  Genome-wide SNP importance", loc="left")
    fig.suptitle(f"Explainable AI: {t}", fontweight="bold", fontsize=12); plt.tight_layout()
    plt.savefig(f"/content/FigSHAP_{t}.png", bbox_inches="tight"); plt.show()
    print(f"{t} done")

# ---- SHAP stability: do top SNPs stay the same when a family is left out? ----
stab = []
for t in TRAITS:
    y = df[t].to_numpy(float); top_all = set(np.argsort(-np.abs(shap_values(X, y)).mean(0))[:20])
    for f in big_fams:
        k = families != f; top_f = set(np.argsort(-np.abs(shap_values(X[k], y[k])).mean(0))[:20])
        stab.append(dict(trait=t, left_out=f, overlap=len(top_all & top_f) / len(top_all | top_f)))
stab = pd.DataFrame(stab)
fig, ax = plt.subplots(figsize=(5, 3.8))
for i, t in enumerate(TRAITS):
    v = stab[stab.trait == t].overlap
    ax.boxplot(v, positions=[i], widths=0.5, showfliers=False, medianprops=dict(color="k"))
    ax.scatter(i + np.random.default_rng(i).uniform(-0.12, 0.12, len(v)), v, color=C[i], s=22, zorder=3)
ax.set_xticks(range(3), TRAITS); ax.set_ylim(0, 1); ax.set_ylabel("Overlap of top-20 SNPs (Jaccard)")
ax.set_title("SHAP stability when a family is left out"); plt.tight_layout()
plt.savefig("/content/FigB_SHAP_stability.png", bbox_inches="tight"); plt.show()

with pd.ExcelWriter("/content/SHAP_results.xlsx") as w:
    pd.concat(top_tables).round(5).to_excel(w, sheet_name="top20_SNPs", index=False)
    stab.round(3).to_excel(w, sheet_name="stability", index=False)
    res.to_excel(w, sheet_name="model_accuracy", index=False)
print("Saved figures (PNG) and /content/SHAP_results.xlsx")

## Cell 3 — Conformal prediction intervals

In [ ]:
# ===== CELL 3: Conformal prediction intervals (run after Cells 1-2, same session) =====
from sklearn.model_selection import GroupKFold

def conformal(Xtr, ytr, gtr, Xte, design):
    """Cross-conformal interval for GBLUP. design = 'Random' or 'Family-aware'."""
    split = GroupKFold(5).split(Xtr, ytr, gtr) if design == "Family-aware" else KFold(5, shuffle=True, random_state=1).split(Xtr)
    res = np.zeros(len(ytr))
    for a, b in split:
        res[b] = np.abs(ytr[b] - fit_predict("GBLUP", Xtr[a], ytr[a], Xtr[b]))
    pred = fit_predict("GBLUP", Xtr, ytr, Xte)
    q = {lvl: np.sort(res)[int(np.ceil(lvl * (len(res) + 1))) - 1] for lvl in (0.8, 0.9)}
    return pred, q

rows, line_rows = [], []
for t in TRAITS:
    y = df[t].to_numpy(float)
    for design in ["Random", "Family-aware"]:
        # (a) random 5-fold CV
        for tr, te in KFold(5, shuffle=True, random_state=1).split(X):
            pred, q = conformal(X[tr], y[tr], families[tr], X[te], design)
            for lvl in (0.8, 0.9):
                rows.append(dict(trait=t, test="Random CV", calibration=design, nominal=lvl,
                                 covered=np.sum(np.abs(y[te] - pred) <= q[lvl]), n=len(te), width=2 * q[lvl]))
        # (b) new family
        for f in big_fams:
            te = np.where(families == f)[0]; tr = np.where(families != f)[0]
            pred, q = conformal(X[tr], y[tr], families[tr], X[te], design)
            for lvl in (0.8, 0.9):
                rows.append(dict(trait=t, test="New family", calibration=design, nominal=lvl,
                                 covered=np.sum(np.abs(y[te] - pred) <= q[lvl]), n=len(te), width=2 * q[lvl]))
            for i, j in enumerate(te):
                line_rows.append(dict(trait=t, family=f, calibration=design, observed=y[j], predicted=pred[i], half90=q[0.9]))
    print(t, "done")

cov = pd.DataFrame(rows).groupby(["trait", "test", "calibration", "nominal"], as_index=False).agg(
    covered=("covered", "sum"), n=("n", "sum"), width=("width", "mean"))
cov["coverage"] = (cov.covered / cov.n).round(3)
cov["width_SD"] = cov.apply(lambda r: r.width / df[r.trait].std(), axis=1).round(2)
cov = cov.drop(columns=["covered", "n"])
print(cov.to_string(index=False))
lines_ci = pd.DataFrame(line_rows)

# ---- Figure C: coverage (a-c) + one new family example (d-e) ----
CC = {"Random": C[0], "Family-aware": C[3]}; MK = {"Random CV": "o", "New family": "s"}
fig = plt.figure(figsize=(15, 8)); g = fig.add_gridspec(2, 3, height_ratios=[1, 1.1])
for i, t in enumerate(TRAITS):
    ax = fig.add_subplot(g[0, i])
    ax.fill_between([0.75, 0.95], [0.70, 0.90], [0.80, 1.0], color="#EEEEEE"); ax.plot([0.75, 0.95], [0.75, 0.95], "k--", lw=0.8)
    for _, r in cov[cov.trait == t].iterrows():
        ax.scatter(r.nominal + (0.006 if r.calibration == "Family-aware" else -0.006), r.coverage,
                   marker=MK[r.test], color=CC[r.calibration], s=55, edgecolor="white")
    ax.set_xticks([0.8, 0.9]); ax.set_xlabel("Target coverage"); ax.set_ylabel("Observed coverage"); ax.set_title(t)
fig.axes[2].legend(handles=[plt.Line2D([], [], marker="o", ls="", c="grey", label="Random CV"),
                            plt.Line2D([], [], marker="s", ls="", c="grey", label="New family"),
                            plt.Line2D([], [], marker="o", ls="", c=C[0], label="Random calibration"),
                            plt.Line2D([], [], marker="o", ls="", c=C[3], label="Family-aware calibration")],
                   fontsize=7, loc="lower right")
f0 = big_fams[0]
for k, design in enumerate(["Random", "Family-aware"]):
    ax = fig.add_subplot(g[1, k])
    d = lines_ci[(lines_ci.trait == "FLW") & (lines_ci.family == f0) & (lines_ci.calibration == design)].sort_values("predicted")
    xx = np.arange(len(d)); ins = (np.abs(d.observed - d.predicted) <= d.half90).to_numpy()
    ax.vlines(xx, d.predicted - d.half90, d.predicted + d.half90, color=CC[design], alpha=0.35, lw=4)
    ax.scatter(xx, d.predicted, s=10, color=CC[design], label="prediction")
    ax.scatter(xx[ins], d.observed[ins], s=16, marker="D", color="k", label="observed (inside)")
    ax.scatter(xx[~ins], d.observed[~ins], s=26, marker="X", color=C[3], label="observed (outside)")
    ax.set_title(f"{design} calibration, family {f0}: 90% coverage = {ins.mean():.0%}")
    ax.set_xlabel("Lines sorted by prediction"); ax.set_ylabel("Days to flowering"); ax.legend(fontsize=7)
ax = fig.add_subplot(g[1, 2]); s = cov[(cov.test == "New family") & (cov.nominal == 0.9)]
for k, design in enumerate(["Random", "Family-aware"]):
    v = s[s.calibration == design].set_index("trait").loc[TRAITS]
    b = ax.bar(np.arange(3) + (k - 0.5) * 0.38, v.width_SD, 0.38, color=CC[design], label=design)
    ax.bar_label(b, labels=[f"{c:.0%}" for c in v.coverage], fontsize=7)
ax.set_xticks(range(3), TRAITS); ax.set_ylabel("90% interval width (trait SD)"); ax.set_ylim(0, s.width_SD.max() * 1.3)
ax.set_title("Width vs coverage (labels) in new families"); ax.legend(fontsize=7, loc="upper left")
fig.suptitle("Conformal prediction intervals: do they hold for a new family?", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/FigC_conformal.png", bbox_inches="tight"); plt.show()

with pd.ExcelWriter("/content/SHAP_results.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    cov.to_excel(w, sheet_name="conformal_coverage", index=False)
print("Saved /content/FigC_conformal.png and added 'conformal_coverage' sheet to SHAP_results.xlsx")

## Cell 4 — Additional figures

In [ ]:
# ===== CELL 4: Modern figures (run after Cells 1-3, same session, ~2 min) =====
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from sklearn.manifold import TSNE
from matplotlib.patches import Patch
fam_order = list(pd.Series(families).value_counts().index)
FCOL = {f: plt.get_cmap("tab20")(i % 20) for i, f in enumerate(fam_order)}
TCMAP = {"FLW": "viridis", "GY": "YlGn", "PH": "magma_r"}

# ---------- Fig 1: Circular genomic tree (UPGMA on IBS) + trait rings ----------
D = np.vstack([np.abs(X - X[i]).mean(1) / 2 for i in range(len(X))])  # 1 - IBS distance (memory-light)
Lk = linkage(squareform(D, checks=False), method="average")
dn = dendrogram(Lk, no_plot=True); leaves = dn["leaves"]; n_l = len(leaves)
ymax = max(max(d) for d in dn["dcoord"])
ang = lambda x: 2 * np.pi * (x - 5) / (10 * n_l) * 0.97                # leaf positions -> angle
rad = lambda h: 1 - h / ymax                                           # height -> radius (leaves at r=1)
fig = plt.figure(figsize=(10, 10)); ax = fig.add_subplot(projection="polar")
for xs, ys in zip(dn["icoord"], dn["dcoord"]):
    a = ang(np.array(xs)); r = rad(np.array(ys))
    ax.plot([a[0], a[0]], [r[0], r[1]], c="#555555", lw=0.4)
    ax.plot([a[3], a[3]], [r[3], r[2]], c="#555555", lw=0.4)
    th = np.linspace(a[1], a[2], 30); ax.plot(th, np.full(30, r[1]), c="#555555", lw=0.4)
theta = ang(np.arange(n_l) * 10 + 5); w = 2 * np.pi / n_l * 0.97
ax.bar(theta, 0.06, width=w, bottom=1.02, color=[FCOL[families[i]] for i in leaves], lw=0)   # family ring
for k, t in enumerate(TRAITS):                                                              # trait rings
    v = df[t].to_numpy(float)[leaves]; v = (v - v.min()) / (v.max() - v.min())
    ax.bar(theta, 0.06, width=w, bottom=1.10 + 0.08 * k, color=plt.get_cmap(TCMAP[t])(v), lw=0)
ax.set_ylim(0, 1.36); ax.axis("off")
ax.legend(handles=[Patch(color=FCOL[f], label=f) for f in fam_order[:12]], loc="center left",
          bbox_to_anchor=(1.02, 0.5), fontsize=7, title="Family (largest 12)")
ax.set_title("Circular UPGMA tree of 353 lines (IBS)\nrings, inner→outer: family, FLW, GY, PH",
             fontweight="bold", pad=20)
plt.savefig("/content/Fig4_circular_tree.png", bbox_inches="tight"); plt.show()

# ---------- Fig 2: Raincloud of per-family accuracy (new family) ----------
fam_r = []
for t in TRAITS:
    y = df[t].to_numpy(float)
    for name in ["GBLUP", "XGBoost"]:
        for f in big_fams:
            k = families == f
            fam_r.append(dict(trait=t, model=name, family=f,
                              r=stats.pearsonr(fit_predict(name, X[~k], y[~k], X[k]), y[k])[0]))
fam_r = pd.DataFrame(fam_r)
fig, axs = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for i, t in enumerate(TRAITS):
    ax = axs[i]
    for j, name in enumerate(["GBLUP", "XGBoost"]):
        v = fam_r[(fam_r.trait == t) & (fam_r.model == name)].r.to_numpy(); col = [C[0], C[3]][j]
        vp = ax.violinplot(v, positions=[j], widths=0.8, showextrema=False)
        for b in vp["bodies"]:
            b.set_facecolor(col); b.set_alpha(0.35); p = b.get_paths()[0]; p.vertices[:, 0] = np.clip(p.vertices[:, 0], j, None)
        ax.boxplot(v, positions=[j - 0.1], widths=0.1, showfliers=False, medianprops=dict(color="k"))
        ax.scatter(j - 0.28 + np.random.default_rng(j).uniform(-0.05, 0.05, len(v)), v, s=22, color=col, zorder=3)
    ax.axhline(0, c="k", ls=":", lw=0.7); ax.set_xticks([0, 1], ["GBLUP", "XGBoost"]); ax.set_title(t)
axs[0].set_ylabel("Accuracy in each new family (r)")
fig.suptitle("Raincloud: prediction accuracy varies widely between new families", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/Fig5_raincloud.png", bbox_inches="tight"); plt.show()

# ---------- Fig 3 (Fig S1): observed vs predicted in new families ----------
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for i, t in enumerate(TRAITS):
    y = df[t].to_numpy(float); pr = np.full(len(y), np.nan)
    for f in big_fams:
        k = families == f; pr[k] = fit_predict("GBLUP", X[~k], y[~k], X[k])
    ax = axs[i]; within = []
    for f in big_fams:
        k = families == f
        ax.scatter(pr[k], y[k], s=10, color=FCOL[f], alpha=0.45, lw=0)                    # lines
        b, a0 = np.polyfit(pr[k], y[k], 1); xx = np.linspace(pr[k].min(), pr[k].max(), 10)
        ax.plot(xx, a0 + b * xx, color=FCOL[f], lw=1.6)                                      # within-family trend
        ax.scatter(pr[k].mean(), y[k].mean(), s=90, color=FCOL[f], edgecolor="k", lw=0.8, zorder=4,
                   label=f if i == 0 else None)                                              # family mean
        within.append(stats.pearsonr(pr[k], y[k])[0])
    s = ~np.isnan(pr); lo, hi = min(pr[s].min(), y[s].min()), max(pr[s].max(), y[s].max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.7); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
    ax.text(0.03, 0.97, f"overall r = {stats.pearsonr(pr[s], y[s])[0]:.2f}\nmean within-family r = {np.mean(within):.2f}",
            transform=ax.transAxes, va="top", fontsize=9, bbox=dict(fc="white", ec="none", alpha=0.8))
    ax.set_xlabel(f"Predicted {t} (new family, GBLUP)"); ax.set_ylabel(f"Observed {t}"); ax.set_title(t)
fig.legend(loc="center right", fontsize=8, title="Family", bbox_to_anchor=(1.01, 0.5))
fig.suptitle("Observed vs predicted in new families: big dots = family means, lines = within-family trend",
             fontweight="bold"); plt.tight_layout(rect=[0, 0, 0.94, 1])
plt.savefig("/content/FigS1_obs_vs_pred.png", bbox_inches="tight"); plt.show()

# ---------- Fig 4: t-SNE genomic landscape coloured by family and by each trait ----------
emb = TSNE(n_components=2, perplexity=30, random_state=1, init="pca").fit_transform(X)
fig, axs = plt.subplots(1, 4, figsize=(18, 4.4))
axs[0].scatter(emb[:, 0], emb[:, 1], c=[FCOL[f] for f in families], s=12, lw=0); axs[0].set_title("Family")
for i, t in enumerate(TRAITS):
    sc = axs[i + 1].scatter(emb[:, 0], emb[:, 1], c=df[t], cmap=TCMAP[t], s=12, lw=0)
    plt.colorbar(sc, ax=axs[i + 1], fraction=0.046); axs[i + 1].set_title(t)
for a in axs: a.set_xticks([]); a.set_yticks([]); a.set_xlabel("t-SNE 1"); a.set_ylabel("t-SNE 2")
fig.suptitle("Genomic landscape (t-SNE of 941 SNPs)", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/FigS2_tSNE.png", bbox_inches="tight"); plt.show()

# ---------- Fig 5: Ridgeline (joy plot) of traits per family ----------
fams10 = [f for f in fam_order if (families == f).sum() >= 10]
fig, axs = plt.subplots(1, 3, figsize=(14, 6), sharey=True)
for i, t in enumerate(TRAITS):
    ax = axs[i]; v_all = df[t].to_numpy(float); grid = np.linspace(v_all.min(), v_all.max(), 200)
    for k, f in enumerate(fams10[::-1]):
        v = v_all[families == f]; dens = stats.gaussian_kde(v)(grid); dens = dens / dens.max() * 1.6
        ax.fill_between(grid, k, k + dens, color=FCOL[f], alpha=0.75, lw=0)
        ax.plot(grid, k + dens, color="white", lw=0.8)
        ax.plot([np.median(v)] * 2, [k, k + 0.6], c="k", lw=0.8)
    ax.set_yticks(range(len(fams10)), fams10[::-1]); ax.set_xlabel(t); ax.spines["left"].set_visible(False)
fig.suptitle("Ridgeline: trait distributions by family (line = median)", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/FigS3_ridgeline.png", bbox_inches="tight"); plt.show()

# ---------- Fig 6: SHAP dependence of the top SNP, split by family ----------
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for i, t in enumerate(TRAITS):
    y = df[t].to_numpy(float); S = shap_values(X, y); j = np.argmax(np.abs(S).mean(0)); ax = axs[i]
    for q, f in enumerate(fams10):
        k = families == f
        ax.scatter(X[k, j] + (q - len(fams10) / 2) * 0.03, S[k, j], s=10, color=FCOL[f], alpha=0.8, lw=0)
    ax.axhline(0, c="grey", lw=0.6); ax.set_xticks([0, 1, 2], ["0", "1", "2"])
    ax.set_xlabel(f"{snp_cols[j]} (minor allele dosage)"); ax.set_ylabel("SHAP value"); ax.set_title(t)
fig.suptitle("SHAP dependence of the top SNP per trait, coloured by family", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/FigS4_SHAP_dependence.png", bbox_inches="tight"); plt.show()

# ---------- Fig 7: Forest plot of per-family accuracy (GBLUP) with 95% CI ----------
fig, axs = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for i, t in enumerate(TRAITS):
    d = fam_r[(fam_r.trait == t) & (fam_r.model == "GBLUP")].set_index("family").loc[big_fams]
    nfam = np.array([(families == f).sum() for f in big_fams])
    z = np.arctanh(np.clip(d.r, -0.99, 0.99)); se = 1 / np.sqrt(nfam - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se); yy = np.arange(len(big_fams))
    axs[i].hlines(yy, lo, hi, color="#888888"); axs[i].scatter(d.r, yy, s=nfam * 2, color=C[0], zorder=3)
    pooled = np.tanh(np.sum(z * (nfam - 3)) / np.sum(nfam - 3))
    axs[i].axvline(pooled, c=C[3], ls="--", lw=1); axs[i].axvline(0, c="k", lw=0.5)
    axs[i].set_title(f"{t}  (pooled r = {pooled:.2f}, dashed)"); axs[i].set_xlabel("Accuracy in new family (r, 95% CI)")
axs[0].set_yticks(yy, [f"{f} (n={(families == f).sum()})" for f in big_fams]); axs[0].invert_yaxis()
fig.suptitle("Forest plot: GBLUP accuracy per new family (dot size = family size)", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/FigS5_forest.png", bbox_inches="tight"); plt.show()
fam_r.round(3).to_csv("/content/per_family_accuracy.csv", index=False)
print("Saved 7 figures + per_family_accuracy.csv")

## Cell 5 — Robustness, statistics and gene annotation

In [ ]:
# ===== CELL 5: Robustness + gene annotation (run after Cells 1-4, same session, ~4 min) =====
import requests, time
t_start = time.time()
REPS = 10                                          # repeats of 5-fold CV
def r_(a, b): return stats.pearsonr(a, b)[0] if np.std(a) > 0 and np.std(b) > 0 else np.nan

# ---------- A. Repeated random CV (10 x 5-fold), within-family, LOFO, parent-disjoint LOFO ----------
parents = {f: set(str(df.loc[families == f, "parents"].iloc[0]).split(" ; ")) for f in set(families)}
def shares_parent(a, b): return a != b and len(parents[a] & parents[b]) > 0
fam10 = [f for f in set(families) if (families == f).sum() >= 10]
rob = []
for t in TRAITS:
    y = df[t].to_numpy(float)
    for name in ["GBLUP", "XGBoost"]:
        for rep in range(REPS):                                            # random CV + within-family
            oof = np.zeros(len(y))
            for tr, te in KFold(5, shuffle=True, random_state=rep).split(X):
                oof[te] = fit_predict(name, X[tr], y[tr], X[te])
                rob.append(dict(trait=t, model=name, scheme="Random CV", unit=f"rep{rep}", r=r_(oof[te], y[te])))
            for f in fam10:
                k = families == f
                rob.append(dict(trait=t, model=name, scheme="Within-family (random CV)", unit=f"{f}_rep{rep}", family=f, r=r_(oof[k], y[k])))
        for f in big_fams:                                                 # LOFO and parent-disjoint LOFO
            te = families == f
            tr_pd = np.array([(g != f) and not shares_parent(f, g) for g in families])
            rob.append(dict(trait=t, model=name, scheme="LOFO", unit=f, family=f,
                            r=r_(fit_predict(name, X[~te], y[~te], X[te]), y[te])))
            rob.append(dict(trait=t, model=name, scheme="Parent-disjoint LOFO", unit=f, family=f, n_train=int(tr_pd.sum()),
                            r=r_(fit_predict(name, X[tr_pd], y[tr_pd], X[te]), y[te])))
    print(f"{t} done ({(time.time()-t_start)/60:.1f} min)")
rob = pd.DataFrame(rob)
# within-family: average the 10 repeats so each family counts once
wf = rob[rob.scheme == "Within-family (random CV)"].groupby(["trait", "model", "family"], as_index=False).r.mean()
wf["scheme"] = "Within-family (random CV)"; wf["unit"] = wf.family
rob2 = pd.concat([rob[rob.scheme != "Within-family (random CV)"], wf])
SCH = ["Random CV", "Within-family (random CV)", "LOFO", "Parent-disjoint LOFO"]
summary = rob2.groupby(["trait", "model", "scheme"]).r.agg(["mean", "std", "count"]).round(3).reset_index()
summary["scheme"] = pd.Categorical(summary.scheme, SCH); summary = summary.sort_values(["trait", "model", "scheme"])
print("\n=== Accuracy (mean ± SD) ==="); print(summary.to_string(index=False))

# ---------- B. Statistical tests (paired over families, Wilcoxon signed-rank, Holm-adjusted) ----------
tests = []
for t in TRAITS:
    for name in ["GBLUP", "XGBoost"]:
        p = rob2[(rob2.trait == t) & (rob2.model == name) & (rob2.scheme != "Random CV")].pivot_table(index="family", columns="scheme", values="r")
        for a, b in [("Within-family (random CV)", "LOFO"), ("LOFO", "Parent-disjoint LOFO")]:
            d = p[[a, b]].dropna()
            tests.append(dict(trait=t, model=name, comparison=f"{a} vs {b}", n_families=len(d),
                              mean_diff=round((d[a] - d[b]).mean(), 3), p=stats.wilcoxon(d[a], d[b]).pvalue))
    lo = rob2[(rob2.trait == t) & (rob2.scheme == "LOFO")].pivot_table(index="family", columns="model", values="r").dropna()
    tests.append(dict(trait=t, model="XGBoost vs GBLUP", comparison="LOFO", n_families=len(lo),
                      mean_diff=round((lo.XGBoost - lo.GBLUP).mean(), 3), p=stats.wilcoxon(lo.XGBoost, lo.GBLUP).pvalue))
tests = pd.DataFrame(tests)
o = np.argsort(tests.p.values); adj = np.empty(len(o)); run = 0
for i, j in enumerate(o): run = max(run, (len(o) - i) * tests.p.values[j]); adj[j] = min(1, run)
tests["p_holm"] = adj; tests["p"] = tests.p.round(4); tests["p_holm"] = tests.p_holm.round(4)
print("\n=== Wilcoxon tests ==="); print(tests.to_string(index=False))

# ---------- C. Sensitivity: remove near-duplicates / possible mislabels (GBLUP, 10x5 CV + LOFO) ----------
sets = {"All lines": np.ones(len(df), bool),
        "Near-duplicates removed": ~df["drop_in_sensitivity"].astype(bool).to_numpy(),
        "Possible mislabels removed": ~df["flag_possible_mislabel"].astype(bool).to_numpy()}
sens = []
for t in TRAITS:
    for lab, keep in sets.items():
        Xs, ys, fs = X[keep], df[t].to_numpy(float)[keep], families[keep]
        cv = [r_(fit_predict("GBLUP", Xs[tr], ys[tr], Xs[te]), ys[te])
              for rep in range(REPS) for tr, te in KFold(5, shuffle=True, random_state=rep).split(Xs)]
        lf = [r_(fit_predict("GBLUP", Xs[fs != f], ys[fs != f], Xs[fs == f]), ys[fs == f]) for f in big_fams if (fs == f).sum() > 5]
        sens.append(dict(trait=t, data=lab, n_lines=int(keep.sum()), randomCV_r=round(np.mean(cv), 3),
                         randomCV_sd=round(np.std(cv), 3), LOFO_r=round(np.nanmean(lf), 3)))
sens = pd.DataFrame(sens); print("\n=== Sensitivity (GBLUP) ==="); print(sens.to_string(index=False))

# ---------- D. Figure: four validation schemes with error bars ----------
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
SC = dict(zip(SCH, [C[0], C[2], C[1], C[3]]))
for i, t in enumerate(TRAITS):
    for j, name in enumerate(["GBLUP", "XGBoost"]):
        for k, s in enumerate(SCH):
            v = rob2[(rob2.trait == t) & (rob2.model == name) & (rob2.scheme == s)].r
            yy = j + (k - 1.5) * 0.18
            axs[i].errorbar(v.mean(), yy, xerr=v.std(), fmt="o", color=SC[s], ms=7, capsize=3, label=s if (i == 0 and j == 0) else None)
    axs[i].axvline(0, c="k", ls=":", lw=0.6); axs[i].set_yticks([0, 1], ["GBLUP", "XGBoost"]); axs[i].invert_yaxis()
    axs[i].set_title(t); axs[i].set_xlabel("Prediction accuracy (r, mean ± SD)")
fig.legend(loc="upper center", ncol=4, fontsize=8, bbox_to_anchor=(0.5, 1.06))
plt.tight_layout(); plt.savefig("/content/Fig6_validation_schemes.png", bbox_inches="tight"); plt.show()

# ---------- E. Genes near the top SHAP SNPs (live lookup in Ensembl Plants, IRGSP-1.0) ----------
WIN = 200_000                                       # search window: +/- 200 kb
KNOWN = {"Ghd8": "Os08g0174500", "Hd1": "Os06g0275000", "Ghd7": "Os07g0261200", "Hd3a": "Os06g0157700",
         "RFT1": "Os06g0157500", "Ehd1": "Os10g0463400", "sd1": "Os01g0883800"}
def ens(url):
    r = requests.get("https://rest.ensembl.org" + url, headers={"Content-Type": "application/json"}, timeout=30)
    r.raise_for_status(); return r.json()
genes_out = []
try:
    print("\n=== Known flowering / height genes (positions from Ensembl) ===")
    known_pos = {}
    for g, gid in KNOWN.items():
        try:
            j = ens(f"/lookup/id/{gid}?species=oryza_sativa"); known_pos[g] = (int(j["seq_region_name"]), j["start"], j["end"])
            print(f"{g:6s} {gid}  chr{j['seq_region_name']}:{j['start']:,}-{j['end']:,}")
        except Exception as e:
            print(f"{g}: lookup failed ({e})")
    top = pd.concat(top_tables).drop_duplicates("SNP")
    for _, s in top.iterrows():
        near = [f"{g} ({abs(int(s.pos) - (p[1] + p[2]) // 2) / 1e3:.0f} kb)" for g, p in known_pos.items()
                if p[0] == int(s.chrom) and abs(int(s.pos) - (p[1] + p[2]) // 2) <= 1_000_000]
        hits = ens(f"/overlap/region/oryza_sativa/{int(s.chrom)}:{max(1, int(s.pos) - WIN)}-{int(s.pos) + WIN}?feature=gene")
        hits = sorted(hits, key=lambda h: abs((h["start"] + h["end"]) // 2 - int(s.pos)))
        genes_out.append(dict(trait=s.trait, SNP=s.SNP, mean_abs_SHAP=round(s.mean_abs_SHAP, 4),
                              known_gene_within_1Mb="; ".join(near),
                              n_genes_within_200kb=len(hits),
                              nearest_genes="; ".join(f"{h['id']}{' ' + h.get('external_name', '') if h.get('external_name') else ''}"
                                                      for h in hits[:5])))
        time.sleep(0.1)
    genes_out = pd.DataFrame(genes_out)
    print("\n=== Top SHAP SNPs near known genes ===")
    print(genes_out[genes_out.known_gene_within_1Mb != ""][["trait", "SNP", "known_gene_within_1Mb"]].to_string(index=False))
except Exception as e:
    print("Gene lookup failed (internet/Ensembl problem):", e, "\nThe rest of the results are saved anyway.")
    genes_out = pd.DataFrame()

# ---------- Save everything into one Excel file ----------
with pd.ExcelWriter("/content/Robustness_results.xlsx") as w:
    summary.to_excel(w, sheet_name="accuracy_summary", index=False)
    tests.to_excel(w, sheet_name="wilcoxon_tests", index=False)
    sens.to_excel(w, sheet_name="sensitivity", index=False)
    rob2.round(4).to_excel(w, sheet_name="accuracy_raw", index=False)
    if len(genes_out): genes_out.to_excel(w, sheet_name="SHAP_SNP_genes", index=False)
print(f"\nSaved /content/Robustness_results.xlsx and Fig6_validation_schemes.png  ({(time.time()-t_start)/60:.1f} min)")

## Cell 5b — Gene lookup with automatic retries (rerun if Ensembl was busy in Cell 5)

In [ ]:
# ===== Gene lookup near top SHAP SNPs (Ensembl Plants, IRGSP-1.0) — with retries =====
import requests, time
WIN = 200_000                                       # search window: +/- 200 kb
KNOWN = {"Ghd8": "Os08g0174500", "Hd1": "Os06g0275000", "Ghd7": "Os07g0261200", "Hd3a": "Os06g0157700",
         "RFT1": "Os06g0157500", "Ehd1": "Os10g0463400", "sd1": "Os01g0883800"}

def ens(url, tries=5):
    for k in range(tries):
        try:
            r = requests.get("https://rest.ensembl.org" + url, headers={"Content-Type": "application/json"}, timeout=40)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:                                   # rate limit: wait as told
                time.sleep(float(r.headers.get("Retry-After", 2)))
        except requests.RequestException:
            pass
        time.sleep(2 * (k + 1))                                        # wait 2, 4, 6, 8 s and retry
    return None

print("=== Known flowering / height genes ===")
known_pos = {}
for g, gid in KNOWN.items():
    j = ens(f"/lookup/id/{gid}?species=oryza_sativa")
    if j:
        known_pos[g] = (int(j["seq_region_name"]), j["start"], j["end"])
        print(f"{g:6s} {gid}  chr{j['seq_region_name']}:{j['start']:,}-{j['end']:,}")
    else:
        print(f"{g:6s} {gid}  lookup failed after retries")

genes_out = []
top = pd.concat(top_tables).drop_duplicates("SNP").reset_index(drop=True)
for n_, s in top.iterrows():
    c, p = int(s.chrom), int(s.pos)
    near = [f"{g} ({abs(p - (q[1] + q[2]) // 2) / 1e3:.0f} kb)" for g, q in known_pos.items()
            if q[0] == c and abs(p - (q[1] + q[2]) // 2) <= 1_000_000]
    hits = ens(f"/overlap/region/oryza_sativa/{c}:{max(1, p - WIN)}-{p + WIN}?feature=gene")
    if hits is None:
        nearest, n_genes = "lookup failed", None
    else:
        hits = sorted(hits, key=lambda h: abs((h["start"] + h["end"]) // 2 - p))
        nearest = "; ".join(f"{h['id']}" + (f" ({h['external_name']})" if h.get("external_name") else "")
                            + f" {abs((h['start'] + h['end']) // 2 - p) / 1e3:.0f}kb" for h in hits[:5])
        n_genes = len(hits)
    genes_out.append(dict(trait=s.trait, SNP=s.SNP, mean_abs_SHAP=round(s.mean_abs_SHAP, 4),
                          known_gene_within_1Mb="; ".join(near), n_genes_within_200kb=n_genes,
                          nearest_5_genes=nearest))
    print(f"\r{n_ + 1}/{len(top)} SNPs looked up", end="")
    time.sleep(0.2)
genes_out = pd.DataFrame(genes_out)

print("\n\n=== Top SHAP SNPs within 1 Mb of a known gene ===")
print(genes_out[genes_out.known_gene_within_1Mb != ""][["trait", "SNP", "mean_abs_SHAP", "known_gene_within_1Mb"]].to_string(index=False))
print(f"\nFailed SNP lookups: {(genes_out.nearest_5_genes == 'lookup failed').sum()} of {len(genes_out)}")

with pd.ExcelWriter("/content/Robustness_results.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    genes_out.to_excel(w, sheet_name="SHAP_SNP_genes", index=False)
print("Added sheet 'SHAP_SNP_genes' to /content/Robustness_results.xlsx")

## Cell 7 — Candidate-gene figure (Ghd8 haplotype and sd1)

In [ ]:
# ===== CELL 7: Candidate-gene figure — Ghd8 region (chr8) and sd1 region (chr1) =====
GHD8 = 4_333_467; SD1 = 38_383_927                       # gene midpoints (Ensembl, IRGSP-1.0)
s1, s2, s3 = "chr08:4190544", "chr08:4511882", "chr01:38632196"
hap = df[s1].astype(int).astype(str) + "-" + df[s2].astype(int).astype(str)
HAP = {"0-0": "Ref–Ref", "0-2": "Ref–Alt", "2-2": "Alt–Alt", "2-0": "Alt–Ref"}
Fd = pd.get_dummies(families, drop_first=True).astype(float).values

def within_family_effect(dummies, y):                     # effect adjusted for family (fixed effect)
    Xm = np.c_[dummies, np.ones(len(y)), Fd]; b = np.linalg.lstsq(Xm, y, rcond=None)[0]
    e = y - Xm @ b; dfree = len(y) - np.linalg.matrix_rank(Xm); V = e @ e / dfree * np.linalg.pinv(Xm.T @ Xm)
    k = dummies.shape[1]; se = np.sqrt(np.diag(V)[:k]); return b[:k], se, 2 * stats.t.sf(np.abs(b[:k] / se), dfree)

fig, ax = plt.subplots(1, 4, figsize=(18, 4.3), gridspec_kw={"width_ratios": [1.4, 1, 1, 0.8]})
# a: SHAP importance along chr8, Ghd8 marked
S = shap_values(X, df["FLW"].to_numpy(float)); imp = np.abs(S).mean(0)
m8 = (chrom == 8) & (pos > 2e6) & (pos < 7e6)
ax[0].vlines(pos[m8] / 1e6, 0, imp[m8], color="#9AB7D3", lw=2); ax[0].scatter(pos[m8] / 1e6, imp[m8], s=25, color=C[0], zorder=3)
ax[0].axvline(GHD8 / 1e6, color=C[3], ls="--", lw=1.2); ax[0].text(GHD8 / 1e6, imp[m8].max() * 1.02, " Ghd8", color=C[3], fontstyle="italic")
ax[0].axvspan(4.190544, 4.511882, color=C[3], alpha=0.08)
ax[0].set_xlabel("Chromosome 8 position (Mb)"); ax[0].set_ylabel("Mean |SHAP| (FLW)"); ax[0].set_title("a  SHAP importance near Ghd8", loc="left")
# b, c: trait by haplotype, with within-family effects
order = ["0-0", "0-2", "2-2", "2-0"]; D = pd.get_dummies(hap)[["2-0", "2-2", "0-2"]].astype(float).values
for i, t in enumerate(["FLW", "PH"]):
    y = df[t].to_numpy(float); b, se, p = within_family_effect(D, y)
    data = [y[hap == h] for h in order]
    ax[i + 1].boxplot(data, widths=0.5, showfliers=False, medianprops=dict(color="k"))
    for j, v in enumerate(data):
        ax[i + 1].scatter(j + 1 + np.random.default_rng(j).uniform(-0.15, 0.15, len(v)), v, s=8, alpha=0.5,
                          color=C[3] if order[j] == "2-0" else C[0], lw=0)
    ax[i + 1].set_xticks(range(1, 5), [f"{HAP[h]}\n(n={len(d)})" for h, d in zip(order, data)], fontsize=8)
    ax[i + 1].set_ylabel(t); ax[i + 1].set_title(f"{'bc'[i]}  {t} by chr8 haplotype", loc="left")
    ax[i + 1].text(0.02, 0.02, f"Alt–Ref vs Ref–Ref, within families:\n{b[0]:+.1f} (SE {se[0]:.1f}), p = {p[0]:.0e}",
                   transform=ax[i + 1].transAxes, fontsize=8, va="bottom", bbox=dict(fc="white", ec="none", alpha=0.85))
    ax[i + 1].set_xlabel(f"Haplotype {s1.split(':')[1]} – {s2.split(':')[1]}")
# d: sd1-region SNP and plant height
y = df["PH"].to_numpy(float); g = df[s3].astype(int).to_numpy()
b, se, p = within_family_effect((g[:, None] / 2).astype(float), y)
ax[3].boxplot([y[g == 0], y[g == 2]], widths=0.5, showfliers=False, medianprops=dict(color="k"))
for j, a in enumerate([0, 2]):
    v = y[g == a]; ax[3].scatter(j + 1 + np.random.default_rng(j).uniform(-0.15, 0.15, len(v)), v, s=8, alpha=0.5, color=C[j * 3], lw=0)
ax[3].set_xticks([1, 2], [f"Ref\n(n={(g == 0).sum()})", f"Alt\n(n={(g == 2).sum()})"])
ax[3].set_ylabel("PH"); ax[3].set_xlabel(f"{s3} ({abs(int(s3.split(':')[1]) - SD1) / 1e3:.0f} kb from sd1)")
ax[3].set_title("d  PH near sd1", loc="left")
ax[3].text(0.02, 0.02, f"within families:\n{b[0]:+.1f} cm (SE {se[0]:.1f}), p = {p[0]:.0e}", transform=ax[3].transAxes, fontsize=8, bbox=dict(fc="white", ec="none", alpha=0.85))
fig.suptitle("Top SHAP markers lie near known heading-date and height genes", fontweight="bold"); plt.tight_layout()
plt.savefig("/content/Fig7_candidate_genes.png", bbox_inches="tight"); plt.show()

## Cell 6 — Download results

In [ ]:
# ===== CELL 6: Download all results to your device =====
import glob
from google.colab import files
for f in sorted(glob.glob("/content/*.png") + glob.glob("/content/*.xlsx") + glob.glob("/content/*.csv")):
    files.download(f)